# 🧠 Phase 2 — Analyse de Sentiment

**Projet : African Media Intelligence AI**  
**Auteur : Esmel Amari (Phil)**  
**Description :** Ce notebook applique l'analyse de sentiment sur les articles collectés en Phase 1, en utilisant un modèle multilingue HuggingFace adapté au français et à l'anglais.

---

### Modèle utilisé
**`cardiffnlp/twitter-xlm-roberta-base-sentiment`**
- Multilingue (français ✅, anglais ✅, 100+ langues)
- Entraîné sur 198M tweets
- Labels : **Negative / Neutral / Positive**
- Score de confiance entre 0 et 1


## 0. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import os
import time
import logging
import warnings
from pathlib import Path
from datetime import datetime

# HuggingFace Transformers
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# ── Chemins ────────────────────────────────────────────────────────
BASE_DIR  = Path('..').resolve()
RAW_DIR   = BASE_DIR / 'data' / 'raw'
PROC_DIR  = BASE_DIR / 'data' / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)

# ── Device (GPU si disponible) ─────────────────────────────────────
DEVICE = 0 if torch.cuda.is_available() else -1
print(f"✅ Device : {'GPU (CUDA)' if DEVICE == 0 else 'CPU'}")

# ── Modèle ─────────────────────────────────────────────────────────
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
print(f"📦 Modèle : {MODEL_NAME}")

✅ Device : CPU
📦 Modèle : cardiffnlp/twitter-xlm-roberta-base-sentiment


## 1. Chargement des Données

In [2]:
# ── Charger le fichier CSV le plus récent du dossier raw ───────────
csv_files = sorted(RAW_DIR.glob('articles_raw_*.csv'), reverse=True)

if not csv_files:
    raise FileNotFoundError(
        "Aucun fichier raw trouvé. Exécutez d'abord le notebook 01_data_collection.ipynb"
    )

latest_file = csv_files[0]
df = pd.read_csv(latest_file, encoding='utf-8-sig')

print(f"✅ Fichier chargé : {latest_file.name}")
print(f"   → {df.shape[0]} articles, {df.shape[1]} colonnes")
print(f"\n📋 Aperçu :")
df[['source', 'titre', 'langue', 'nb_mots']].head(3)

✅ Fichier chargé : articles_raw_20260512_2014.csv
   → 121 articles, 14 colonnes

📋 Aperçu :


,source,titre,langue,nb_mots
0,AllAfrica,"Kenya: MPs Probe Missing 27,000 Tonnes of Impo...",en,35
1,AllAfrica,Nigeria: Over 100 Killed in Zamfara Market Air...,en,36
2,AllAfrica,Sudan: UN Warns Sudan War Entering 'Deadlier P...,en,49


## 2. Chargement du Modèle

In [3]:
print(f"⏳ Chargement du modèle {MODEL_NAME}...")
print("   (Premier chargement : ~2-3 min selon connexion)\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

sentiment_pipeline = pipeline(
    task            = "sentiment-analysis",
    model           = model,
    tokenizer       = tokenizer,
    device          = DEVICE,
    return_all_scores = True,  # Retourne les scores pour les 3 labels
    truncation      = True,
    max_length      = 512
)

print("✅ Modèle chargé avec succès")

# Test rapide
test_texts = [
    "L'économie africaine connaît une croissance exceptionnelle.",
    "La crise politique plonge le pays dans le chaos.",
    "Le gouvernement a publié un nouveau rapport trimestriel."
]
print("\n🔬 Test du modèle :")
for t in test_texts:
    result = sentiment_pipeline(t[:128])[0]
    best   = max(result, key=lambda x: x['score'])
    print(f"  '{t[:60]}...'")
    print(f"  → {best['label']} ({best['score']:.2%})\n")

⏳ Chargement du modèle cardiffnlp/twitter-xlm-roberta-base-sentiment...
   (Premier chargement : ~2-3 min selon connexion)



2026-05-12 20:22:22,793 [INFO] HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-12 20:22:22,899 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cardiffnlp/twitter-xlm-roberta-base-sentiment/f2f1202b1bdeb07342385c3f807f9c07cd8f5cf8/config.json "HTTP/1.1 200 OK"
2026-05-12 20:22:23,311 [INFO] HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-05-12 20:22:23,580 [INFO] HTTP Request: HEAD https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-05-12 20:22:23,591 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-12 20:22:23,824 [INFO] HTTP Request: GET https://huggingface.co/api/m

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Modèle chargé avec succès

🔬 Test du modèle :


TypeError: string indices must be integers, not 'str'

## 3. Fonction d'Analyse de Sentiment

In [4]:
# Mapping labels → français
LABEL_MAP = {
    'Negative' : 'Négatif',
    'Neutral'  : 'Neutre',
    'Positive' : 'Positif',
    'negative' : 'Négatif',
    'neutral'  : 'Neutre',
    'positive' : 'Positif',
    'NEGATIVE' : 'Négatif',
    'NEUTRAL'  : 'Neutre',
    'POSITIVE' : 'Positif',
}

def analyze_sentiment_batch(texts: list, batch_size: int = 16) -> list:
    """
    Analyse le sentiment d'une liste de textes par batch.

    Retourne :
        Liste de dicts :
        {
          'sentiment'       : str   (Positif / Neutre / Négatif),
          'score_positif'   : float,
          'score_neutre'    : float,
          'score_negatif'   : float,
          'score_confiance' : float  (score du label dominant)
        }
    """
    results = []
    total   = len(texts)

    for i in range(0, total, batch_size):
        batch = texts[i:i + batch_size]

        # Tronquer chaque texte à 512 tokens (limite du modèle)
        batch_clean = [str(t)[:1500] if t else "" for t in batch]

        try:
            batch_results = sentiment_pipeline(batch_clean)

            for res in batch_results:
                scores = {r['label'].lower(): r['score'] for r in res}

                # Récupérer les 3 scores
                s_pos = scores.get('positive', 0.0)
                s_neu = scores.get('neutral',  0.0)
                s_neg = scores.get('negative', 0.0)

                # Label dominant
                dominant = max(res, key=lambda x: x['score'])
                label_fr = LABEL_MAP.get(dominant['label'], dominant['label'])

                results.append({
                    'sentiment'       : label_fr,
                    'score_positif'   : round(s_pos, 4),
                    'score_neutre'    : round(s_neu, 4),
                    'score_negatif'   : round(s_neg, 4),
                    'score_confiance' : round(dominant['score'], 4)
                })

        except Exception as e:
            logger.error(f"Erreur batch {i}-{i+batch_size}: {e}")
            for _ in batch:
                results.append({
                    'sentiment'       : 'Neutre',
                    'score_positif'   : 0.0,
                    'score_neutre'    : 1.0,
                    'score_negatif'   : 0.0,
                    'score_confiance' : 0.0
                })

        # Progression
        pct = min(100, round((i + batch_size) / total * 100))
        print(f"\r  Progression : {pct}% ({min(i+batch_size, total)}/{total})", end='')

    print()
    return results


print("✅ Fonction batch définie")

✅ Fonction batch définie


## 4. Application sur le Dataset

In [5]:
# ── Texte d'analyse : titre + résumé (plus représentatif que contenu full) ──
texts = (df['titre'] + '. ' + df['resume'].fillna('')).tolist()

print(f"🔄 Analyse de sentiment sur {len(texts)} articles...")
t0 = time.time()

sentiment_results = analyze_sentiment_batch(texts, batch_size=16)

elapsed = round(time.time() - t0, 1)
print(f"✅ Terminé en {elapsed}s ({round(len(texts)/elapsed, 1)} articles/sec)")

# ── Intégration dans le DataFrame ─────────────────────────────────
sentiment_df = pd.DataFrame(sentiment_results)
df = pd.concat([df.reset_index(drop=True), sentiment_df], axis=1)

print(f"\n📊 Distribution des sentiments :")
print(df['sentiment'].value_counts().to_string())
print(f"\n📊 Score de confiance moyen : {df['score_confiance'].mean():.2%}")

🔄 Analyse de sentiment sur 121 articles...


2026-05-12 20:24:21,344 [ERROR] Erreur batch 0-16: string indices must be integers, not 'str'


  Progression : 13% (16/121)

2026-05-12 20:24:45,904 [ERROR] Erreur batch 16-32: string indices must be integers, not 'str'


  Progression : 26% (32/121)

...

  Progression : 79% (96/121)

2026-05-12 20:26:42,162 [ERROR] Erreur batch 96-112: string indices must be integers, not 'str'


  Progression : 93% (112/121)

2026-05-12 20:26:54,767 [ERROR] Erreur batch 112-128: string indices must be integers, not 'str'


  Progression : 100% (121/121)
✅ Terminé en 181.1s (0.7 articles/sec)

📊 Distribution des sentiments :
sentiment
Neutre    121

📊 Score de confiance moyen : 0.00%


## 5. Analyse par Source & Langue

In [6]:
print("=== SENTIMENT PAR SOURCE ===")
pivot_source = pd.crosstab(
    df['source'],
    df['sentiment'],
    normalize='index'
).round(3) * 100
print(pivot_source.to_string())

print("\n=== SENTIMENT PAR LANGUE ===")
pivot_langue = pd.crosstab(
    df['langue'],
    df['sentiment'],
    normalize='index'
).round(3) * 100
print(pivot_langue.to_string())

print("\n=== TOP 5 ARTICLES LES PLUS POSITIFS ===")
cols = ['source', 'titre', 'sentiment', 'score_positif']
print(df.nlargest(5, 'score_positif')[cols].to_string())

print("\n=== TOP 5 ARTICLES LES PLUS NÉGATIFS ===")
print(df.nlargest(5, 'score_negatif')[cols].to_string())

=== SENTIMENT PAR SOURCE ===
sentiment          Neutre
source                   
AllAfrica           100.0
BBC Afrique         100.0
Jeune Afrique       100.0
Le Monde Afrique    100.0
The Africa Report   100.0

=== SENTIMENT PAR LANGUE ===
sentiment  Neutre
langue           
en          100.0
fr          100.0

=== TOP 5 ARTICLES LES PLUS POSITIFS ===
      source                                                                                                                       titre sentiment  score_positif
0  AllAfrica                                     Kenya: MPs Probe Missing 27,000 Tonnes of Imported Sugar Declared Unfit for Consumption    Neutre            0.0
1  AllAfrica                                                                        Nigeria: Over 100 Killed in Zamfara Market Airstrike    Neutre            0.0
2  AllAfrica  Sudan: UN Warns Sudan War Entering 'Deadlier Phase' As Drone Strikes Kill Hundreds, Attacks Spread Across Multiple Regions    Neutre            0

## 6. Sauvegarde

In [7]:
output_file = PROC_DIR / f"articles_sentiment_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✅ Sauvegardé : {output_file}")
print(f"   → {df.shape[0]} articles, {df.shape[1]} colonnes")
print(f"   → {round(output_file.stat().st_size / 1024, 1)} KB")
print(f"\nColonnes ajoutées : sentiment, score_positif, score_neutre, score_negatif, score_confiance")

✅ Sauvegardé : C:\Users\E682\Desktop\data\processed\articles_sentiment_20260512_2027.csv
   → 121 articles, 19 colonnes
   → 138.1 KB

Colonnes ajoutées : sentiment, score_positif, score_neutre, score_negatif, score_confiance


---

## ✅ Phase 2 Terminée

**Ce qui a été accompli :**
- ✅ Modèle multilingue XLM-RoBERTa chargé
- ✅ Sentiment analysé sur tous les articles (batch processing)
- ✅ 5 nouvelles colonnes : `sentiment`, `score_positif`, `score_neutre`, `score_negatif`, `score_confiance`
- ✅ Distribution par source et par langue calculée

**Prochaine étape :** `03_topic_modeling.ipynb` — Détection des sujets tendances avec BERTopic
